# 12 Hotspot localization evaluation

Uses Grad-CAM/model-input dimensions rather than guessed image paths, preventing coordinate-frame mismatch.

In [1]:
from pathlib import Path
import os, json, shutil, zipfile, math, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image


# 12. HOTSPOT LOCALIZATION EVALUATION


print("RUNNING SCRIPT 12 PATHOLOGICAL-ONLY VERSION 2026-06-07")

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

BASE_DIR = Path(
    os.environ.get(
        "THERMO_BASE_DIR",
        "/content" if Path("/content").exists() else "/mnt/data"
    )
)

PROJECT_NAME = os.environ.get("THERMO_PROJECT_NAME", "project_thermography_equine")
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_ROOT = PROJECT_ROOT / "data"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
PROCESSED_DIR = DATA_ROOT / "processed"
CLEAN_IMAGE_DIR = PROCESSED_DIR / "clean_images"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
GRADCAM_DIR = OUTPUT_ROOT / "gradcam"
CASE_REVIEW_DIR = OUTPUT_ROOT / "case_review"
LOCALIZATION_QC_DIR = CASE_REVIEW_DIR / "hotspot_localization_qc"

for d in [
    PROJECT_ROOT, DATA_ROOT, SPLIT_DATA_DIR, PROCESSED_DIR, CLEAN_IMAGE_DIR,
    OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, GRADCAM_DIR,
    CASE_REVIEW_DIR, LOCALIZATION_QC_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

CAM_NATIVE_GRID = int(os.environ.get("THERMO_CAM_NATIVE_GRID", 7))
BORDER_PX = int(os.environ.get("THERMO_BORDER_PX", 5))
HEATMAP_QUANTILES = [0.80, 0.90]


def count_images(root):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(
        1 for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTS
        and ".ipynb_checkpoints" not in p.parts
    )


def auto_unzip_image_archives():
    search_dirs = [
        PROJECT_ROOT, DATA_ROOT, PROCESSED_DIR, CLEAN_IMAGE_DIR,
        SPLIT_DATA_DIR, Path("/content"), Path("/mnt/data")
    ]

    zip_files = []

    for root in search_dirs:
        if root.exists():
            zip_files.extend([
                p for p in root.rglob("*.zip")
                if p.is_file() and ".ipynb_checkpoints" not in p.parts
            ])

    zip_files = sorted(set(zip_files))
    extracted = []

    for zpath in zip_files:
        lname = zpath.name.lower()

        if "clean" in lname or "224" in lname or "processed" in lname:
            out_dir = CLEAN_IMAGE_DIR
        elif "split" in lname or "dataset" in lname:
            out_dir = SPLIT_DATA_DIR
        elif "image" in lname or "img" in lname:
            out_dir = CLEAN_IMAGE_DIR
        else:
            out_dir = zpath.parent

        try:
            with zipfile.ZipFile(zpath, "r") as z:
                members = [
                    m for m in z.namelist()
                    if Path(m).suffix.lower() in IMAGE_EXTS
                    and "__MACOSX" not in m
                ]

                if members:
                    z.extractall(out_dir)
                    extracted.append({
                        "zip": str(zpath),
                        "to": str(out_dir),
                        "n_image_members": len(members),
                    })

        except Exception as e:
            warnings.warn(f"Could not extract {zpath}: {e}")

    report = {
        "auto_unzip_executed": True,
        "n_zip_files_found": len(zip_files),
        "n_archives_extracted": len(extracted),
        "n_images_in_clean_image_dir": count_images(CLEAN_IMAGE_DIR),
        "n_images_in_dataset_split": count_images(SPLIT_DATA_DIR),
        "extracted_archives": extracted,
    }

    (REPORTS_DIR / "localization_auto_unzip_report.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )

    print(json.dumps(report, indent=2))


def find_file(target_name, patterns=None):
    patterns = [target_name] + (patterns or [])

    preferred = [
        CONFIG_DIR / target_name,
        REPORTS_DIR / target_name,
        TABLES_DIR / target_name,
        GRADCAM_DIR / target_name,
        OUTPUT_ROOT / target_name,
        PROJECT_ROOT / target_name,
        BASE_DIR / target_name,
        Path("/content") / target_name,
        Path("/mnt/data") / target_name,
    ]

    for p in preferred:
        if p.exists() and p.is_file():
            return p

    hits = []

    for root in [
        CONFIG_DIR, REPORTS_DIR, TABLES_DIR, GRADCAM_DIR,
        OUTPUT_ROOT, PROJECT_ROOT, BASE_DIR, Path("/content"), Path("/mnt/data")
    ]:
        if not root.exists():
            continue

        for pat in patterns:
            hits.extend([
                x for x in root.rglob(pat)
                if x.is_file() and ".ipynb_checkpoints" not in x.parts
            ])

    if not hits:
        return None

    src = sorted(hits, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    dst = CONFIG_DIR / target_name

    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)

    return dst


def read_required(target_name, patterns=None):
    p = find_file(target_name, patterns)

    if p is None:
        raise FileNotFoundError(
            f"Missing required file: {target_name}. Expected it under {PROJECT_ROOT}."
        )

    return pd.read_csv(p)


def normalize_image_name(x):
    return Path(str(x)).name


def point_in_box(x, y, b):
    return bool(b[0] <= x <= b[2] and b[1] <= y <= b[3])


def box_area(b):
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])


def inter(a, b):
    return max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(
        0, min(a[3], b[3]) - max(a[1], b[1])
    )


def iou(a, b):
    u = box_area(a) + box_area(b) - inter(a, b)
    return float(inter(a, b) / u) if u else np.nan


def cam_cell(mx, my, w, h, grid=7):
    cw = w / grid
    ch = h / grid
    cx = int(min(grid - 1, max(0, math.floor(mx / cw))))
    cy = int(min(grid - 1, max(0, math.floor(my / ch))))
    return (cx * cw, cy * ch, (cx + 1) * cw, (cy + 1) * ch), cx, cy


def heatmap_shape(path):
    try:
        a = np.load(path)
        return float(a.shape[1]), float(a.shape[0])
    except Exception:
        return np.nan, np.nan


def scaled_bbox(r):
    w = float(r["gradcam_image_width"])
    h = float(r["gradcam_image_height"])

    sx = w / float(r["image_width"])
    sy = h / float(r["image_height"])

    x1 = float(r["xtl"]) * sx
    x2 = float(r["xbr"]) * sx
    y1 = float(r["ytl"]) * sy
    y2 = float(r["ybr"]) * sy

    x1, x2 = sorted([max(0, min(w, x1)), max(0, min(w, x2))])
    y1, y2 = sorted([max(0, min(h, y1)), max(0, min(h, y2))])

    return x1, y1, x2, y2, sx, sy


def dist_target(r):
    if pd.notna(r.get("point_x_scaled", np.nan)) and pd.notna(r.get("point_y_scaled", np.nan)):
        tx = float(r["point_x_scaled"])
        ty = float(r["point_y_scaled"])
        typ = "expert_point"
    else:
        tx = (r["bbox_xtl_scaled"] + r["bbox_xbr_scaled"]) / 2
        ty = (r["bbox_ytl_scaled"] + r["bbox_ybr_scaled"]) / 2
        typ = "bbox_center"

    d = math.hypot(float(r["gradcam_max_x"]) - tx, float(r["gradcam_max_y"]) - ty)
    return d, typ, tx, ty


def heatmap_metrics(path, bbox, w, h):
    outs = []

    try:
        cam = np.load(path).squeeze().astype(float)

        if cam.shape != (int(h), int(w)):
            cam_norm = (cam - cam.min()) / (cam.max() - cam.min() + 1e-12)
            cam = np.asarray(
                Image.fromarray((cam_norm * 255).astype(np.uint8)).resize(
                    (int(w), int(h)), Image.BILINEAR
                )
            ) / 255.0

        mask = np.zeros((int(h), int(w)), dtype=bool)

        x1, y1, x2, y2 = [int(round(v)) for v in bbox]
        x1 = max(0, min(int(w) - 1, x1))
        x2 = max(0, min(int(w), x2))
        y1 = max(0, min(int(h) - 1, y1))
        y2 = max(0, min(int(h), y2))

        mask[y1:y2, x1:x2] = True

        for q in HEATMAP_QUANTILES:
            cam_mask = cam >= np.nanquantile(cam, q)
            union = np.logical_or(cam_mask, mask).sum()
            inn = np.logical_and(cam_mask, mask).sum()

            outs.append({
                "heatmap_quantile": q,
                "cam_bbox_iou": float(inn / union) if union else np.nan,
                "cam_bbox_any_overlap": bool(inn > 0),
                "iou_status": "ok",
            })

    except Exception as e:
        for q in HEATMAP_QUANTILES:
            outs.append({
                "heatmap_quantile": q,
                "cam_bbox_iou": np.nan,
                "cam_bbox_any_overlap": np.nan,
                "iou_status": f"failed_{str(e)[:80]}",
            })

    return outs


def summarize(df, group, level):
    return {
        "group": group,
        "level": level,
        "n_rows": int(len(df)),
        "n_images": int(df["image_name"].nunique()),
        "strict_pointing_accuracy": float(df["strict_pointing_hit"].mean()),
        "cam_cell_overlap_accuracy": float(df["cam_cell_bbox_overlap"].mean()),
        "median_distance_px": float(df["distance_px"].median()),
        "mean_distance_px": float(df["distance_px"].mean()),
        "median_cam_cell_bbox_iou": float(df["cam_cell_bbox_iou"].median()),
        "border_max_fraction": float(df["gradcam_max_on_border"].mean()),
    }



# Load required files


auto_unzip_image_archives()

gradcam = read_required(
    "gradcam_summary.csv",
    ["gradcam_summary*.csv", "*gradcam*summary*.csv"]
)

bboxes = read_required(
    "hotspot_bboxes.csv",
    ["hotspot_bboxes*.csv", "*bbox*.csv", "*hotspot*box*.csv"]
)

points_path = find_file(
    "hotspot_points.csv",
    ["hotspot_points*.csv", "*hotspot*point*.csv"]
)

points = pd.read_csv(points_path) if points_path else pd.DataFrame()



# Standardize names and numeric columns


for df in [gradcam, bboxes, points]:
    if not df.empty and "image_name" in df.columns:
        df["image_name"] = df["image_name"].map(normalize_image_name)

required_gradcam_cols = ["image_name", "gradcam_max_x", "gradcam_max_y"]
missing_gradcam = [c for c in required_gradcam_cols if c not in gradcam.columns]
if missing_gradcam:
    raise KeyError(f"gradcam_summary.csv missing columns: {missing_gradcam}")

required_bbox_cols = ["image_name", "xtl", "ytl", "xbr", "ybr", "image_width", "image_height"]
missing_bbox = [c for c in required_bbox_cols if c not in bboxes.columns]
if missing_bbox:
    raise KeyError(f"hotspot_bboxes.csv missing columns: {missing_bbox}")

for col in [
    "gradcam_max_x", "gradcam_max_y", "gradcam_image_width",
    "gradcam_image_height", "label_binary", "cnn_predicted_label_binary"
]:
    if col in gradcam.columns:
        gradcam[col] = pd.to_numeric(gradcam[col], errors="coerce")

for col in ["xtl", "ytl", "xbr", "ybr", "image_width", "image_height", "label_binary"]:
    if col in bboxes.columns:
        bboxes[col] = pd.to_numeric(bboxes[col], errors="coerce")

if not points.empty:
    for col in ["x", "y", "image_width", "image_height"]:
        if col in points.columns:
            points[col] = pd.to_numeric(points[col], errors="coerce")


# Prepare Grad-CAM rows


gradcam_eval = gradcam.copy()

if "gradcam_status" in gradcam_eval.columns:
    gradcam_eval = gradcam_eval[
        gradcam_eval["gradcam_status"].astype(str).str.lower().eq("ok")
    ].copy()

gradcam_eval = gradcam_eval.dropna(subset=["gradcam_max_x", "gradcam_max_y"]).copy()

if "gradcam_image_width" not in gradcam_eval.columns:
    gradcam_eval["gradcam_image_width"] = np.nan

if "gradcam_image_height" not in gradcam_eval.columns:
    gradcam_eval["gradcam_image_height"] = np.nan

for i, r in gradcam_eval.iterrows():
    if (
        pd.isna(r.get("gradcam_image_width", np.nan))
        or pd.isna(r.get("gradcam_image_height", np.nan))
    ):
        w, h = heatmap_shape(r.get("gradcam_heatmap_path", None))
        gradcam_eval.loc[i, "gradcam_image_width"] = w if np.isfinite(w) else 224
        gradcam_eval.loc[i, "gradcam_image_height"] = h if np.isfinite(h) else 224



# Prepare expert boxes


valid = (
    bboxes["xbr"].notna()
    & bboxes["xtl"].notna()
    & bboxes["ybr"].notna()
    & bboxes["ytl"].notna()
    & bboxes["image_width"].notna()
    & bboxes["image_height"].notna()
    & (bboxes["xbr"] > bboxes["xtl"])
    & (bboxes["ybr"] > bboxes["ytl"])
    & (bboxes["image_width"] > 0)
    & (bboxes["image_height"] > 0)
)

if "bbox_valid" in bboxes.columns:
    valid &= bboxes["bbox_valid"].astype(str).str.lower().isin(["true", "1", "yes", "y"])

bboxes_eval = bboxes[valid].copy()



# Merge Grad-CAM with expert boxes

expert_all = gradcam_eval.merge(
    bboxes_eval,
    on="image_name",
    how="inner",
    suffixes=("_gradcam", "_bbox")
)

if expert_all.empty:
    diagnostics = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "error": "No evaluable Grad-CAM x expert bbox rows after image_name merge.",
        "n_gradcam_eval_rows": int(len(gradcam_eval)),
        "n_bbox_eval_rows": int(len(bboxes_eval)),
        "n_gradcam_images": int(gradcam_eval["image_name"].nunique()),
        "n_bbox_images": int(bboxes_eval["image_name"].nunique()),
        "example_gradcam_image_names": gradcam_eval["image_name"].head(20).tolist(),
        "example_bbox_image_names": bboxes_eval["image_name"].head(20).tolist(),
    }

    (CONFIG_DIR / "hotspot_localization_merge_failure.json").write_text(
        json.dumps(diagnostics, indent=2), encoding="utf-8"
    )

    raise RuntimeError(json.dumps(diagnostics, indent=2))


# PRIMARY COHORT: PATHOLOGICAL ONLY


label_col = (
    "label_clinical_gradcam"
    if "label_clinical_gradcam" in expert_all.columns
    else "label_clinical"
    if "label_clinical" in expert_all.columns
    else None
)

if label_col is None:
    raise KeyError(
        "No clinical label column found. Expected 'label_clinical_gradcam' or 'label_clinical'."
    )

expert_all["_label_for_localization"] = (
    expert_all[label_col].astype(str).str.lower().str.strip()
)

pathological_values = {
    "pathological", "pathology", "diseased", "disease",
    "positive", "1", "true", "yes"
}

healthy_values = {
    "healthy", "normal", "negative", "0", "false", "no"
}

expert_pathological = expert_all[
    expert_all["_label_for_localization"].isin(pathological_values)
].copy()

expert_healthy_control = expert_all[
    expert_all["_label_for_localization"].isin(healthy_values)
].copy()

cohort_diag = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "label_column_used": label_col,
    "n_merged_rows_before_pathological_filter": int(len(expert_all)),
    "n_merged_images_before_pathological_filter": int(expert_all["image_name"].nunique()),
    "n_pathological_rows_for_primary_localization": int(len(expert_pathological)),
    "n_pathological_images_for_primary_localization": int(expert_pathological["image_name"].nunique()),
    "n_healthy_rows_excluded_from_primary_localization": int(len(expert_healthy_control)),
    "n_healthy_images_excluded_from_primary_localization": int(expert_healthy_control["image_name"].nunique()),
    "reason": (
        "Primary hotspot localization metrics were restricted to pathological cases, "
        "because healthy horses are not expected to have expert-defined inflammatory hotspots."
    ),
}

for outdir in [CONFIG_DIR, REPORTS_DIR]:
    (outdir / "localization_cohort_definition.json").write_text(
        json.dumps(cohort_diag, indent=2), encoding="utf-8"
    )

print(json.dumps(cohort_diag, indent=2))

if expert_pathological.empty:
    raise RuntimeError(
        "No pathological cases available for primary localization analysis after filtering."
    )

expert = expert_pathological.copy()



# Scale expert boxes

scaled = expert.apply(
    lambda r: pd.Series(
        scaled_bbox(r),
        index=[
            "bbox_xtl_scaled", "bbox_ytl_scaled",
            "bbox_xbr_scaled", "bbox_ybr_scaled", "scale_x", "scale_y"
        ],
    ),
    axis=1,
)

expert = pd.concat([expert.reset_index(drop=True), scaled.reset_index(drop=True)], axis=1)



# Optional expert points


if (
    not points.empty
    and {"image_name", "x", "y", "image_width", "image_height"}.issubset(points.columns)
):
    join_cols = ["image_name"]

    if "expert" in points.columns and "expert" in expert.columns:
        join_cols.append("expert")

    pts = points[
        join_cols + ["x", "y", "image_width", "image_height"]
    ].rename(
        columns={
            "x": "point_x_original",
            "y": "point_y_original",
            "image_width": "point_image_width",
            "image_height": "point_image_height",
        }
    )

    expert = expert.merge(pts, on=join_cols, how="left")

    expert["point_x_scaled"] = (
        expert["point_x_original"]
        * expert["gradcam_image_width"]
        / expert["point_image_width"]
    )

    expert["point_y_scaled"] = (
        expert["point_y_original"]
        * expert["gradcam_image_height"]
        / expert["point_image_height"]
    )

else:
    expert["point_x_scaled"] = np.nan
    expert["point_y_scaled"] = np.nan



# Localization metrics


rows = []
iou_rows = []

for _, r in expert.iterrows():
    w = float(r["gradcam_image_width"])
    h = float(r["gradcam_image_height"])
    mx = float(r["gradcam_max_x"])
    my = float(r["gradcam_max_y"])

    box = (
        r["bbox_xtl_scaled"],
        r["bbox_ytl_scaled"],
        r["bbox_xbr_scaled"],
        r["bbox_ybr_scaled"],
    )

    cell, cx, cy = cam_cell(mx, my, w, h, CAM_NATIVE_GRID)
    d, typ, tx, ty = dist_target(r)

    out = r.to_dict()

    out.update({
        "analysis_cohort": "pathological_primary_localization",
        "strict_pointing_hit": point_in_box(mx, my, box),
        "cam_native_grid": CAM_NATIVE_GRID,
        "cam_cell_x_index": cx,
        "cam_cell_y_index": cy,
        "cam_cell_xtl": cell[0],
        "cam_cell_ytl": cell[1],
        "cam_cell_xbr": cell[2],
        "cam_cell_ybr": cell[3],
        "cam_cell_bbox_overlap": inter(cell, box) > 0,
        "cam_cell_bbox_iou": iou(cell, box),
        "distance_px": d,
        "distance_target_type": typ,
        "target_x_scaled": tx,
        "target_y_scaled": ty,
        "gradcam_max_on_border": bool(
            mx <= BORDER_PX
            or my <= BORDER_PX
            or mx >= w - 1 - BORDER_PX
            or my >= h - 1 - BORDER_PX
        ),
    })

    rows.append(out)

    for hm in heatmap_metrics(out.get("gradcam_heatmap_path"), box, w, h):
        iou_rows.append({
            "image_name": out.get("image_name"),
            "expert": out.get("expert"),
            "analysis_cohort": "pathological_primary_localization",
            "label_clinical": out.get(label_col),
            **hm,
        })

results = pd.DataFrame(rows)
iou_df = pd.DataFrame(iou_rows)



# Summaries


summary_rows = [summarize(results, "overall_pathological_only", "all")]

if label_col in results.columns:
    for k, s in results.groupby(label_col, dropna=False):
        summary_rows.append(summarize(s, "label_clinical", str(k)))

if "expert" in results.columns:
    for k, s in results.groupby("expert", dropna=False):
        summary_rows.append(summarize(s, "expert", str(k)))

summary = pd.DataFrame(summary_rows)

healthy_control_summary = pd.DataFrame([
    {
        "analysis_cohort": "healthy_negative_control_excluded_from_primary_localization",
        "n_rows": int(len(expert_healthy_control)),
        "n_images": int(expert_healthy_control["image_name"].nunique())
        if not expert_healthy_control.empty else 0,
        "reason": (
            "Healthy horses were excluded from primary hotspot localization metrics "
            "because no inflammatory hotspot is expected by definition."
        ),
    }
])



# Save outputs


results.to_csv(CONFIG_DIR / "pointing_game_and_distance_results.csv", index=False)
results.to_csv(CONFIG_DIR / "gradcam_expert_bbox_merged.csv", index=False)
iou_df.to_csv(CONFIG_DIR / "cam_bbox_iou_results.csv", index=False)
summary.to_csv(CONFIG_DIR / "localization_summary_by_group.csv", index=False)

summary.to_csv(TABLES_DIR / "table_hotspot_localization_summary.csv", index=False)
healthy_control_summary.to_csv(CONFIG_DIR / "healthy_negative_control_summary.csv", index=False)
healthy_control_summary.to_csv(TABLES_DIR / "healthy_negative_control_summary.csv", index=False)

pointing_legacy_cols = [
    "image_name", "strict_pointing_hit", "cam_cell_bbox_overlap",
    "cam_cell_bbox_iou", "distance_px", "gradcam_max_on_border"
]

pointing_legacy = results[
    [c for c in pointing_legacy_cols if c in results.columns]
].copy()

pointing_legacy.to_csv(CONFIG_DIR / "pointing_game_results.csv", index=False)
pointing_legacy.to_csv(TABLES_DIR / "pointing_game_results.csv", index=False)

iou_df.to_csv(CONFIG_DIR / "iou_results.csv", index=False)
iou_df.to_csv(TABLES_DIR / "iou_results.csv", index=False)

point_distance_summary = pd.DataFrame([
    {
        "analysis_cohort": "pathological_primary_localization",
        "n_rows": int(len(results)),
        "n_images": int(results["image_name"].nunique()),
        "median_distance_px": float(results["distance_px"].median()),
        "mean_distance_px": float(results["distance_px"].mean()),
        "border_max_fraction": float(results["gradcam_max_on_border"].mean()),
    }
])

point_distance_summary.to_csv(CONFIG_DIR / "point_distance_summary.csv", index=False)
point_distance_summary.to_csv(TABLES_DIR / "point_distance_summary.csv", index=False)

loc_by_label = summary[summary["group"].eq("label_clinical")].copy()
loc_by_label.to_csv(CONFIG_DIR / "localization_summary_by_label.csv", index=False)
loc_by_label.to_csv(TABLES_DIR / "localization_summary_by_label.csv", index=False)


# Diagnostics and methods text


diag = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "gradcam_summary_found": str(find_file("gradcam_summary.csv", ["gradcam_summary*.csv"])),
    "hotspot_bboxes_found": str(find_file("hotspot_bboxes.csv", ["hotspot_bboxes*.csv"])),
    "hotspot_points_found": str(points_path) if points_path else None,
    "n_gradcam_rows_total": int(len(gradcam)),
    "n_gradcam_rows_evaluable": int(len(gradcam_eval)),
    "n_bbox_rows_total": int(len(bboxes)),
    "n_bbox_rows_evaluable": int(len(bboxes_eval)),
    "n_merged_rows_before_pathological_filter": int(len(expert_all)),
    "n_merged_images_before_pathological_filter": int(expert_all["image_name"].nunique()),
    "n_pathological_eval_rows": int(len(results)),
    "n_pathological_eval_images": int(results["image_name"].nunique()),
    "n_healthy_rows_excluded_from_primary_localization": int(len(expert_healthy_control)),
    "n_healthy_images_excluded_from_primary_localization": int(expert_healthy_control["image_name"].nunique())
    if not expert_healthy_control.empty else 0,
    "coordinate_frame": "gradcam/model_input_size_preferred",
    "cam_native_grid": CAM_NATIVE_GRID,
    "heatmap_quantiles": HEATMAP_QUANTILES,
    "primary_endpoint": "pathological-only hotspot localization",
    "note": (
        "Primary localization metrics were restricted to pathological cases. "
        "Healthy cases were excluded because no expert-defined inflammatory hotspot "
        "is expected in healthy horses."
    ),
}

for outdir in [CONFIG_DIR, REPORTS_DIR]:
    (outdir / "hotspot_localization_evaluation_diagnostics.json").write_text(
        json.dumps(diag, indent=2), encoding="utf-8"
    )

methods_text = (
    "Primary hotspot localization analysis was restricted to pathological cases, "
    "because healthy horses were not expected to have expert-defined inflammatory "
    "hotspots. Expert hotspot boxes were rescaled to the Grad-CAM/model input "
    "coordinate frame before localization analysis. The primary localization "
    "metric was strict pointing-game accuracy, defined as whether the maximum "
    "Grad-CAM activation fell inside the expert hotspot bounding box. Secondary "
    "localization metrics included native CAM-cell overlap, CAM-cell IoU, "
    "Euclidean distance to the hotspot target, border-activation fraction, "
    "and thresholded heatmap-to-box IoU at the 80th and 90th heatmap percentiles. "
    "Healthy cases were excluded from the primary localization endpoint and "
    "summarized separately as a negative-control cohort."
)

for outdir in [CONFIG_DIR, REPORTS_DIR]:
    (outdir / "methods_hotspot_localization_evaluation_text.txt").write_text(
        methods_text + "\n", encoding="utf-8"
    )

print(summary.to_string(index=False))
print(json.dumps(diag, indent=2))

RUNNING SCRIPT 12 PATHOLOGICAL-ONLY VERSION 2026-06-07
{
  "auto_unzip_executed": true,
  "n_zip_files_found": 2,
  "n_archives_extracted": 2,
  "n_images_in_clean_image_dir": 347,
  "n_images_in_dataset_split": 347,
  "extracted_archives": [
    {
      "zip": "/content/clean_images_224x224.zip",
      "to": "/content/project_thermography_equine/data/processed/clean_images",
      "n_image_members": 347
    },
    {
      "zip": "/content/dataset_split-20260605T090743Z-3-001.zip",
      "to": "/content/project_thermography_equine/data/dataset_split",
      "n_image_members": 347
    }
  ]
}
{
  "created_utc": "2026-06-07T08:01:11.420605+00:00",
  "label_column_used": "label_clinical_gradcam",
  "n_merged_rows_before_pathological_filter": 38,
  "n_merged_images_before_pathological_filter": 19,
  "n_pathological_rows_for_primary_localization": 26,
  "n_pathological_images_for_primary_localization": 13,
  "n_healthy_rows_excluded_from_primary_localization": 12,
  "n_healthy_images_exclud